# Main composite figure

The five connected panels are prepared together because they share data, model caches, and final layout constraints.

In [2]:
%%capture
from pathlib import Path

if Path.cwd().name == "notebooks_altair":
    %cd ..

%load_ext autoreload
%autoreload 2

In [3]:
%%capture
import json
import logging
import os
from datetime import datetime
from pathlib import Path
from typing import Any

import altair as alt
import joblib
import polars as pl
from dotenv import load_dotenv

from src.data.database_manager import DatabaseManager
from src.log_config import configure_logging
from src.models.data_loader import create_dataloaders
from src.models.data_preparation import (
    expand_feature_list,
    load_data_from_database,
    prepare_data,
)
from src.models.main_config import RANDOM_SEED
from src.models.utils import load_model
from src.plots.averages_over_stimulus_seeds import average_over_stimulus_seeds
from src.plots.model_performance import get_model_predictions
from src.plots.model_performance_per_participant import analyze_per_participant
from src.plots.plot_seed_stability import (
    extract_final_test_accuracies,
    load_results,
)
from src.plots_altair import (
    compose_figure_altair,
    plot_accuracy_distributions,
    plot_correlation_heatmap,
    plot_grand_averaged_signals,
    plot_participant_accuracies,
    plot_roc_curves,
    style_figure,
)

configure_logging(
    stream_level=logging.WARNING,
    ignore_libs=("Comm", "bokeh", "tornado", "matplotlib"),
)
pl.Config.set_tbl_rows(12)
alt.data_transformers.disable_max_rows()

## Shared grand averages for panels A and B

In [4]:
database = DatabaseManager()
with database:
    explore_data = database.get_trials("Explore_Data", exclude_problematic=True)

explore_data = explore_data.rename(
    {
        "rating": "pain_rating",
        "pupil": "pupil_diameter",
    }
)
signals = [
    "temperature",
    "pain_rating",
    "pupil_diameter",
    "eda_tonic",
    "eda_phasic",
    "heart_rate",
    "mouth_open",
]
averages = average_over_stimulus_seeds(
    explore_data,
    signals,
    scaling="min_max",
    bin_size=0.1,
    confidence_level=0.95,
)

## Cached model results for panels C and E

In [5]:
feature_lists = expand_feature_list(
    [
        ["eda_raw"],
        ["heart_rate"],
        ["pupil"],
        ["eda_raw", "heart_rate"],
        ["eda_raw", "pupil"],
        ["eda_raw", "heart_rate", "pupil"],
        ["face"],
        ["face", "eda_raw", "heart_rate", "pupil"],
        ["eeg"],
        ["eeg", "eda_raw"],
        ["eeg", "face", "eda_raw", "heart_rate", "pupil"],
    ]
)

In [6]:
class InferenceCache:
    def __init__(self, cache_dir: Path = Path(".cache/model_inference")):
        self.cache_dir = cache_dir
        self.cache_dir.mkdir(parents=True, exist_ok=True)

    def _get_model_timestamp(self, model_path: Path) -> str:
        parts = model_path.name.split("_")
        if len(parts) >= 2:
            return parts[-1].removesuffix(".pt")
        return str(datetime.fromtimestamp(model_path.stat().st_mtime))

    def _get_cache_key(self, feature_list_str: str, cache_type: str, **kwargs) -> str:
        key_parts = [feature_list_str, cache_type]
        for key, value in sorted(kwargs.items()):
            if isinstance(value, (list, tuple)):
                value = "_".join(map(str, value))
            key_parts.append(f"{key}_{value}")
        return "_".join(key_parts)

    def _get_cache_path(self, cache_key: str) -> Path:
        return self.cache_dir / f"{cache_key}.joblib"

    def _is_cache_valid(self, feature_list_str: str, model_path: Path) -> bool:
        timestamp_file = self.cache_dir / f"{feature_list_str}_timestamp.txt"
        if not timestamp_file.exists():
            return False
        return timestamp_file.read_text().strip() == self._get_model_timestamp(
            model_path
        )

    def _update_timestamp(self, feature_list_str: str, model_path: Path) -> None:
        timestamp_file = self.cache_dir / f"{feature_list_str}_timestamp.txt"
        timestamp_file.write_text(self._get_model_timestamp(model_path))

    def get(self, feature_list_str: str, cache_type: str, **kwargs) -> Any:
        cache_key = self._get_cache_key(feature_list_str, cache_type, **kwargs)
        cache_path = self._get_cache_path(cache_key)
        if cache_path.exists():
            try:
                logging.debug("Cache hit: %s", cache_key)
                return joblib.load(cache_path)
            except Exception as error:
                logging.warning("Failed to load cache %s: %s", cache_key, error)
        return None

    def set(self, feature_list_str: str, cache_type: str, data: Any, **kwargs) -> None:
        cache_key = self._get_cache_key(feature_list_str, cache_type, **kwargs)
        cache_path = self._get_cache_path(cache_key)
        joblib.dump(data, cache_path, compress=3)


cache = InferenceCache()

In [7]:
def load_model_and_data(feature_list: list, feature_list_str: str) -> tuple:
    result_path = Path(f"results/experiment_{feature_list_str}/results.json")
    result = json.loads(result_path.read_text())
    model_path = Path(result["overall_best"]["model_path"].replace("\\", "/"))

    (
        model,
        feature_list_loaded,
        sample_duration_ms,
        intervals,
        label_mapping,
        offsets_ms,
    ) = load_model(model_path, device="cpu")
    model_df = load_data_from_database(feature_list)

    _, _, _, _, X_train_val, y_train_val, X_test, y_test = prepare_data(
        df=model_df,
        feature_list=feature_list_loaded,
        sample_duration_ms=sample_duration_ms,
        intervals=intervals,
        label_mapping=label_mapping,
        offsets_ms=offsets_ms,
        random_seed=RANDOM_SEED,
    )
    test_groups = prepare_data(
        df=model_df,
        feature_list=feature_list_loaded,
        sample_duration_ms=sample_duration_ms,
        intervals=intervals,
        label_mapping=label_mapping,
        offsets_ms=offsets_ms,
        random_seed=RANDOM_SEED,
        only_return_test_groups=True,
    )
    _, test_loader = create_dataloaders(
        X_train_val,
        y_train_val,
        X_test,
        y_test,
        batch_size=64,
    )
    model_config = {
        "model_name": model.__class__.__name__,
        "model_path": model_path,
    }
    return model, model_config, test_loader, test_groups


def load_roc_results(cache: InferenceCache) -> dict:
    results = {}
    for feature_list in feature_lists:
        feature_list_str = "_".join(feature_list)
        result_path = Path(f"results/experiment_{feature_list_str}/results.json")
        result = json.loads(result_path.read_text())
        model_path = Path(result["overall_best"]["model_path"].replace("\\", "/"))

        if cache._is_cache_valid(feature_list_str, model_path):
            cached = cache.get(feature_list_str, "test_predictions")
            if cached is not None:
                results[feature_list_str] = (cached["probs"], cached["y_true"])
                continue

        model, model_config, test_loader, _ = load_model_and_data(
            feature_list, feature_list_str
        )
        probabilities, y_true = get_model_predictions(model, test_loader)
        results[feature_list_str] = (probabilities, y_true)
        cache._update_timestamp(feature_list_str, model_config["model_path"])
        cache.set(
            feature_list_str,
            "test_predictions",
            {
                "probs": probabilities,
                "y_true": y_true,
                "model_name": model_config["model_name"],
            },
        )
    return results


def load_participant_results(cache: InferenceCache) -> dict:
    results = {}
    for feature_list in feature_lists:
        feature_list_str = "_".join(feature_list)
        result_path = Path(f"results/experiment_{feature_list_str}/results.json")
        result = json.loads(result_path.read_text())
        model_path = Path(result["overall_best"]["model_path"].replace("\\", "/"))

        if cache._is_cache_valid(feature_list_str, model_path):
            cached = cache.get(feature_list_str, "per_participant_results")
            if cached is not None and not cached.is_empty():
                results[feature_list_str] = cached
                continue

        model, model_config, test_loader, test_groups = load_model_and_data(
            feature_list, feature_list_str
        )
        participant_result = analyze_per_participant(
            model,
            test_loader,
            test_groups,
            threshold=0.5,
        )
        results[feature_list_str] = participant_result
        cache._update_timestamp(feature_list_str, model_config["model_path"])
        cache.set(
            feature_list_str,
            "per_participant_results",
            participant_result,
        )
    return results

## Figure inputs

In [8]:
roc_results = load_roc_results(cache)
participant_results = load_participant_results(cache)
accuracy_distributions = {
    "EDA + HR": extract_final_test_accuracies(
        load_results("results/experiment_eda_raw_heart_rate/stability_results.json")
    ),
    "EDA + HR + Pupil": extract_final_test_accuracies(
        load_results(
            "results/experiment_eda_raw_heart_rate_pupil/stability_results.json"
        )
    ),
}
reference_accuracies = {
    "EDA + HR": 0.756,
    "EDA + HR + Pupil": 0.768,
}

## Connected panels

In [9]:
panel_a = plot_grand_averaged_signals(
    averages,
    signals,
    stimulus_seed=133,
    signal_labels=None,
    signal_colors=None,
    width=390,
    height=200,
    display_step_ms=500,
    y_domain=(-0.05, 1.05),
    line_width=1.8,
    line_opacity=0.95,
    show_ci=True,
    ci_opacity=0.15,
    legend_title="Signal",
    legend_orient="right",
    legend_columns=1,
    title=None,
)
panel_b = plot_correlation_heatmap(
    averages,
    features=None,
    skip_first_n_seconds=20,
    width=200,
    height=200,
    title=None,
)
panel_c = plot_roc_curves(
    roc_results,
    width=200,
    height=200,
    title=None,
)
panel_d = plot_accuracy_distributions(
    accuracy_distributions,
    reference_accuracies,
    bin_width=0.01,
    width=200,
    height=200,
    title=None,
)
panel_e = plot_participant_accuracies(
    participant_results,
    width=800,
    height=200,
    x_label_angle=-45,
    title=None,
)

In [23]:
composite_figure = compose_figure_altair(
    plot_grand_averaged_signals(
        averages,
        signals,
        stimulus_seed=133,
        signal_labels=None,
        signal_colors=None,
        width=390,
        height=245,
        display_step_ms=500,
        y_domain=(0.0, 1.00),
        line_width=1.8,
        line_opacity=0.95,
        show_ci=True,
        ci_opacity=0.15,
        legend_title="Signal",
        legend_orient="right",
        legend_columns=1,
        title=None,
    ),
    plot_correlation_heatmap(
        averages,
        features=None,
        skip_first_n_seconds=20,
        width=245,
        height=245,
        title=None,
    ),
    plot_roc_curves(
        roc_results,
        width=213,
        height=213,
        title=None,
    ),
    plot_accuracy_distributions(
        accuracy_distributions,
        reference_accuracies,
        bin_width=0.01,
        width=213,
        height=213,
        title=None,
    ),
    plot_participant_accuracies(
        participant_results,
        width=880,
        height=250,
        title=None,
        x_label_angle=-30,
    ),
    row_spacing=6,
    middle_row_label_y=-10,
)
style_figure(composite_figure)

alt.VConcatChart(...)

In [20]:
load_dotenv()
FIGURE_DIR = Path(os.environ["FIGURE_DIR"])
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
style_figure(composite_figure).save(FIGURE_DIR / "composite_figure.svg")